In [ ]:
# General
import numpy as np
import pandas as pd
import os
import tqdm

# Read in json files stored at url
import urllib.request, json 


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Census')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config  = os.path.join(path_code, 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')

In [ ]:
with urllib.request.urlopen("https://api.census.gov/data/2022/acs/acs5/variables.json") as url:
    dict_acs = json.load(url)

In [ ]:
dict_acs['variables']

In [ ]:
df_vars = pd.DataFrame.from_dict(dict_acs['variables']).T.reset_index().rename(columns = {'index':'ID'})

df_vars = df_vars[['group', 'ID', 'attributes', 'label', 'concept']].rename(columns = {'group':'Table'})
df_vars['Label_clean'] = df_vars['label'].str.replace('Estimate!!', '')
df_vars['Label_clean'] = df_vars['Label_clean'].str.replace('!!', ' ')
df_vars['Label_clean'] = df_vars['Label_clean'].str.replace(':', '')

df_vars = df_vars.sort_values(['Table', 'ID'])
print(df_vars.shape) # 28193
df_vars.head()

In [ ]:
# df_vars.to_excel(os.path.join(path_config, 'ACS 5-Year Estimates Variable Tables and Labels 2022 JSON.xlsx'), index=False)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
years_to_import = sequence(2009, 2022, 1)
list_df = []

for year in tqdm(years_to_import):
    with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs5/variables.json") as url:
        dict_acs = json.load(url)
    
    df_vars = pd.DataFrame.from_dict(dict_acs['variables']).T.reset_index().rename(columns = {'index':'ID'})
    
    df_vars = df_vars[['group', 'ID', 'attributes', 'label', 'concept']].rename(columns = {'group':'Table'})
    df_vars['Label_clean'] = df_vars['label'].str.replace('Estimate!!', '')
    df_vars['Label_clean'] = df_vars['Label_clean'].str.replace('!!', ' ')
    df_vars['Label_clean'] = df_vars['Label_clean'].str.replace(':', '')
    df_vars['Year'] = year

    list_df.append(df_vars)

df_vars = pd.concat(list_df)
df_vars = df_vars.sort_values(['Table', 'ID', 'Year'], ascending = [True, True, False])
print(df_vars.shape) # 28193
df_vars.head()

In [ ]:
# df_vars.to_excel(os.path.join(path_config, 'ACS 5-Year Estimates Variable Tables and Labels All years JSON.xlsx'), index=False)

Old approach - "censusdata" package only goes through 2019 for some reason

In [ ]:
# # Geographic
# from census import Census
# import censusdata as acs

In [ ]:
# # gather all transportation variables
# # rename columns
# # clean label field

# # df_vars = pd.DataFrame(acs.search('acs5', 2019, 'concept', 'transportation'))
# # df_vars = pd.DataFrame(acs.search('acs5', 2019, 'label', 'Total'))
# # df_vars = pd.DataFrame(acs.search('acs5', 2019, 'label', ''))


# df_vars.columns = ['ID', 'Table Name', 'Label']
# df_vars['Label_clean'] = df_vars['Label'].str.replace('Estimate!!', '')
# df_vars['Label_clean'] = df_vars['Label_clean'].str.replace('!!', ' ')
# df_vars['Label_clean'] = df_vars['Label_clean'].str.replace(':', '')

# # show
# print(df_vars.shape)
# df_vars.head()

In [ ]:
# # Export locally
# df_vars.to_excel(os.path.join(path_config, 'ACS 5-Year Estimates Variable Tables and Labels 2019.xlsx'), index=False)